# Tabla base de discusiones tecnicas desde PDFs

Este cuaderno lee reportes expertos en PDF, extrae fragmentos candidatos y usa un LLM como skill/agente extractor para generar una tabla base con discusiones tecnicas verificables. No usa embeddings, FAISS, Chroma ni bases vectoriales.

La salida final de Excel contiene exactamente estas columnas: `Circuito`, `Fecha inicio`, `Fecha fin`, `Análisis`, `Evidencia`.

## 1. Configuracion inicial

In [1]:
import os
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# Localizar raiz del proyecto y cargar configuracion activa.
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if load_dotenv:
    _env_file = ROOT / ".env"
    if _env_file.exists():
        load_dotenv(_env_file)
        print(f"Config cargada desde: {_env_file.name}")
    else:
        print("Config .env no encontrada; se usara entorno del sistema u Ollama si esta disponible.")

# Parametros principales del usuario
pdf_dir = ROOT / "reports" / "analysis-documents"
fecha_inicio_usuario = "2025-11-01"
fecha_fin_usuario = "2026-04-30"
output_excel = ROOT / "reports" / "analysis-documents" / "tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx"

# Configuracion del LLM, siguiendo el patron de 02_local_uiti_vano_interpretability_v3.ipynb
CALL_LLM = os.getenv("CALL_LLM", "true").strip().lower() not in {"0", "false", "no", "off"}
_configured_provider = os.getenv("LLM_PROVIDER")
if _configured_provider:
    LLM_PROVIDER = _configured_provider
elif os.getenv("GOOGLE_API_KEY", "").strip():
    LLM_PROVIDER = "google"
elif os.getenv("OPENAI_API_KEY", "").strip():
    LLM_PROVIDER = "openai"
else:
    LLM_PROVIDER = "ollama"

if LLM_PROVIDER.lower() == "ollama":
    LLM_MODEL = os.getenv("LLM_MODEL") or os.getenv("OLLAMA_MODEL", "deepseek-r1:32b")
elif LLM_PROVIDER.lower() == "openai":
    LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4.1-mini")
else:
    LLM_MODEL = os.getenv("LLM_MODEL", "gemini-2.5-flash-lite")

LLM_MAX_OUTPUT_TOKENS = int(os.getenv("LLM_MAX_OUTPUT_TOKENS", "8192"))

# Segmentacion y preseleccion
CHUNK_CHARS = 6500
CHUNK_OVERLAP = 800
MAX_FRAGMENTOS = None  # usar un entero pequeno para pruebas rapidas

# Periodos generales conocidos por PDF. Si se deja vacio, el cuaderno intenta inferirlos del texto.
# Formato: {"reporte.pdf": ("YYYY-MM-DD", "YYYY-MM-DD")}
periodos_generales_por_pdf = {}

project_root = ROOT
artifact_root = project_root / "reports" / "interpretability" / "artifacts" / "pdf_discussion_extraction"
Path(output_excel).parent.mkdir(parents=True, exist_ok=True)
artifact_root.mkdir(parents=True, exist_ok=True)

print(f"PDF_DIR     : {pdf_dir}")
print(f"OUTPUT_XLSX : {output_excel}")
print(f"CALL_LLM    : {CALL_LLM}")
print(f"LLM_PROVIDER: {LLM_PROVIDER}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print(f"LLM_MAX_OUT : {LLM_MAX_OUTPUT_TOKENS}")

Config cargada desde: .env
PDF_DIR     : /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/reports/analysis-documents
OUTPUT_XLSX : /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/reports/analysis-documents/tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx
CALL_LLM    : True
LLM_PROVIDER: google
LLM_MODEL   : gemini-2.5-flash-lite
LLM_MAX_OUT : 8192


## 2. Carga de librerias

In [2]:
from __future__ import annotations

from datetime import datetime
import json

import pandas as pd

from chec_local_interpreter.reports.pdf_discussions import (
    PDFDiscussionExtractionSkill,
    chunk_pdf_pages,
    circuito_from_pdf_name,
    detect_report_period,
    extract_pdf_pages,
    extract_json_object,
    is_candidate_fragment,
    iso_fecha,
    validate_llm_row,
)


COLUMNAS_FINALES = ["Circuito", "Fecha inicio", "Fecha fin", "Análisis", "Evidencia"]
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%dT%H%M%S")
run_dir = artifact_root / RUN_TIMESTAMP
run_dir.mkdir(parents=True, exist_ok=True)

## 3. Utilidades de fechas, texto y validacion

In [3]:
# Date, JSON, and validation helpers are imported from `chec_local_interpreter.reports.pdf_discussions`.


## 4. Extraccion de texto y segmentacion

In [ ]:
pdf_paths = sorted(Path(pdf_dir).glob("*.pdf"))
print(f"PDFs encontrados: {len(pdf_paths)}")

all_fragments = []
for pdf_path in pdf_paths:
    circuito_pdf = circuito_from_pdf_name(pdf_path)
    if circuito_pdf is None:
        print(f"{pdf_path.name}: omitido porque el nombre no contiene circuito.")
        continue
    pages = extract_pdf_pages(pdf_path)
    full_text = "\n".join(page.texto for page in pages)
    configured_period = periodos_generales_por_pdf.get(pdf_path.name)
    general_period = configured_period or detect_report_period(full_text)
    fragments = chunk_pdf_pages(pages, general_period, chunk_chars=CHUNK_CHARS, chunk_overlap=CHUNK_OVERLAP)
    candidates = [fragment for fragment in fragments if is_candidate_fragment(fragment)]
    print(f"{pdf_path.name}: circuito={circuito_pdf}, {len(pages)} paginas, {len(fragments)} fragmentos, {len(candidates)} candidatos")
    all_fragments.extend(candidates)

if MAX_FRAGMENTOS is not None:
    all_fragments = all_fragments[:MAX_FRAGMENTOS]

print(f"Fragmentos candidatos totales: {len(all_fragments)}")

PDFs encontrados: 8
AGU23L15.pdf: circuito=AGU23L15, 19 paginas, 7 fragmentos, 7 candidatos
Advertencia: DON23L13.pdf pagina 30 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 31 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 32 no tiene texto extraible.
Advertencia: DON23L13.pdf pagina 33 no tiene texto extraible.
DON23L13.pdf: circuito=DON23L13, 45 paginas, 14 fragmentos, 14 candidatos
INS23L13.pdf: circuito=INS23L13, 12 paginas, 5 fragmentos, 5 candidatos
MAZ23L13.pdf: circuito=MAZ23L13, 23 paginas, 11 fragmentos, 11 candidatos
Advertencia: MNA23L12.pdf pagina 8 no tiene texto extraible.
MNA23L12.pdf: circuito=MNA23L12, 8 paginas, 2 fragmentos, 2 candidatos
SNA23L12_V5.pdf: circuito=SNA23L12_V5, 30 paginas, 11 fragmentos, 11 candidatos
SNA23L15_V7.pdf: circuito=SNA23L15_V7, 21 paginas, 7 fragmentos, 7 candidatos
VMA23L13.pdf: circuito=VMA23L13, 24 paginas, 4 fragmentos, 4 candidatos
Fragmentos candidatos totales: 61


## 5. Skill/agente extractor

In [5]:
SKILL_PATH = project_root / "llm" / "skills" / "pdf_discussion_extraction_01_pdf_discussion_extractor.md"


## 6. Ejecutar extraccion, validar y guardar artefactos

In [6]:
skill = PDFDiscussionExtractionSkill(
    skill_path=SKILL_PATH,
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    call_enabled=CALL_LLM,
    max_output_tokens=LLM_MAX_OUTPUT_TOKENS,
)

rows = []
invalid_records = []
jsonl_path = run_dir / "pdf_discussion_contexts_prompts_outputs.jsonl"

with jsonl_path.open("w", encoding="utf-8") as fh:
    for idx, fragment in enumerate(all_fragments, start=1):
        context = {
            "fecha_inicio_usuario": fecha_inicio_usuario,
            "fecha_fin_usuario": fecha_fin_usuario,
            "nombre_pdf": fragment.nombre_pdf,
            "circuito_pdf": circuito_from_pdf_name(fragment.nombre_pdf),
            "pagina_inicio": fragment.pagina_inicio,
            "pagina_fin": fragment.pagina_fin,
            "periodo_general_informe": fragment.periodo_general_informe,
            "fragmento": fragment.fragmento,
        }
        extraction = skill.extract(context)
        extraction["fragment_index"] = idx
        fh.write(json.dumps(extraction, ensure_ascii=False) + "\n")

        parsed = extraction.get("parsed")
        if not parsed:
            invalid_records.append({**extraction, "validation_errors": ["No se pudo parsear JSON valido"]})
            continue
        circuito_pdf = context["circuito_pdf"]
        if circuito_pdf is None:
            invalid_records.append({**extraction, "validation_errors": ["Nombre de PDF sin circuito"]})
            continue
        parsed["Circuito"] = circuito_pdf
        ok, errors = validate_llm_row(
            parsed,
            fecha_inicio_usuario,
            fecha_fin_usuario,
            final_columns=COLUMNAS_FINALES,
        )
        if not ok:
            invalid_records.append({**extraction, "validation_errors": errors})
            continue

        row = {col: str(parsed[col]).strip() for col in COLUMNAS_FINALES}
        row["Fecha inicio"] = iso_fecha(row["Fecha inicio"])
        row["Fecha fin"] = iso_fecha(row["Fecha fin"])
        rows.append(row)

if invalid_records:
    invalid_path = run_dir / "invalid_llm_outputs.json"
    invalid_path.write_text(json.dumps(invalid_records, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Salidas invalidas o descartadas guardadas en: {invalid_path}")

print(f"Filas validas antes de deduplicar: {len(rows)}")

Salidas invalidas o descartadas guardadas en: /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/reports/interpretability/artifacts/pdf_discussion_extraction/20260712T234736/invalid_llm_outputs.json
Filas validas antes de deduplicar: 13


## 7. Deduplicar y exportar Excel final

In [7]:
df_final = pd.DataFrame(rows, columns=COLUMNAS_FINALES)
if not df_final.empty:
    df_final = df_final.drop_duplicates(subset=COLUMNAS_FINALES).sort_values(
        by=["Circuito", "Fecha inicio", "Fecha fin", "Análisis", "Evidencia"]
    )

df_final = df_final.reindex(columns=COLUMNAS_FINALES)
df_final.to_excel(output_excel, index=False)

print(f"Excel generado: {output_excel}")
print(f"Filas finales: {len(df_final)}")
print(f"Artefactos de trazabilidad: {run_dir}")
df_final.head(20)

Excel generado: /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/reports/analysis-documents/tabla_pdfs_intervalo_2025-11-01_2026-04-30.xlsx
Filas finales: 13
Artefactos de trazabilidad: /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/reports/interpretability/artifacts/pdf_discussion_extraction/20260712T234736


,Circuito,Fecha inicio,Fecha fin,Análisis,Evidencia
0,AGU23L15,2016-01-01,2025-12-21,El circuito AGU23L15 presenta una frecuencia m...,Con la misma información utilizada para el dia...
3,AGU23L15,2025-01-01,2025-12-31,Se identifica una tendencia de degradación pos...,Se identificó una tendencia de degradación pos...
2,AGU23L15,2025-09-23,2025-12-18,Análisis del comportamiento de las señales del...,Se realizó un ejercicio del comportamiento de ...
1,AGU23L15,2025-12-21,2025-12-21,Análisis de confiabilidad post-depuración de d...,Tras depurar la base de datos (eliminando reci...
4,DON23L13,2025-06-01,2026-04-30,Se observa una mejora en la severidad (duració...,se observa una señal de mejora en severidad (h...
5,INS23L13,2025-10-21,2026-03-03,Mantenimientos en C22969 (repotenciación y des...,Tras los mantenimientos del 21 de octubre de 2...
8,MAZ23L13,2025-01-01,2026-02-12,Análisis de causas de eventos no programados e...,Tabla 6-3. Top 10 eventos más severos por dura...
6,MAZ23L13,2025-04-01,2026-05-01,Análisis de confiabilidad del circuito MAZ23L1...,Ventana de análisis (SCADA) 2025-04-01 a 2026-...
7,MAZ23L13,2025-04-04,2026-05-01,Métricas operativas del SCADA del interruptor ...,El análisis del SCADA del interruptor de cabec...
9,SNA23L12_V5,2025-01-01,2026-04-30,Análisis de rayos con potencial afectación a l...,Para los años 2025–2026 se identificaron 24.44...
